# 01 · 云上的仓库：用 Terraform 建 4 个存储桶

## 这一章在整个项目里的位置

```
00 账号安全 ✅ → [01 建仓库 ✅] → 02 代码库工程化 → 03 数据进货 → 04–09 Databricks → 10–12 对外服务和 AI
```

> **这一章你已经跑完了**（10 月 7 日 12:13，AWS 里 21 个资源都在）。这一版只是把讲解按新格式重写，内容没变。再跑一遍是安全的：Terraform 会发现"都已经有了"，什么都不改。

## 背景：为什么需要这一章

00 号给账号装好了锁和报警器，但账号里还是空的。后面每一章都需要**放东西的地方**：86 GB 的原始数据、导出给 API 的结果表、模型文件、给 AI 读的文档。

AWS 上放文件的地方叫 **S3 存储桶**（bucket），可以理解成云上的硬盘文件夹（[基础知识](../../docs/primer_aws_databricks.md) §2.2）。

这一章回答两个问题：
1. **建哪些桶，每个桶怎么保护**：这是 AWS 的内容
2. **用什么方式建**：公司里不在网页上点，而是**写代码描述"应该有什么"**，让工具去建。这个工具叫 **Terraform**

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 建 4 个桶，各管一类东西 | S3 | 后面每一章放东西的地方 |
| 2 | 给每个桶加保护 | S3 设置 + KMS 钥匙 | 数据不泄露、不丢、加密 |
| 3 | 用代码描述以上一切 | Terraform | 能审查、能重建、能核对 |
| 4 | 给自动化流程一个身份 | IAM 角色 + OIDC | 02 号的 GitHub 自动检查不用密码访问 AWS |
| 5 | 先看后做 | `plan` → `apply` | 不会误建、误删 |
| 6 | 去 AWS 网页上看一眼 | AWS 控制台 | 认识这些东西在 AWS 里长什么样 |

**花费：约 1 美元/月**（KMS 钥匙）。桶是空的，不花钱。

---

## 准备

运行下面这一格。它定义一个执行命令的小工具，并确认你现在用的是哪个 AWS 身份。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = Path.cwd().parents[1] if Path.cwd().name == "A_foundation" else Path.cwd()
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"   # Terraform 和 aws 命令都用 00 号登录的身份
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
run("aws sts get-caller-identity --query Arn --output text")

**结果怎么看**：`arn:aws:iam::186972156153:user/pipi-admin`，就是 00 号建的你的日常身份。

> 如果报错说登录过期：在终端运行 `aws login --profile qafi`，然后重新运行这一格。

---

## 第 1 步 · 4 个桶：各放什么，哪一章用

### 在干什么
在 AWS 上建 4 个 S3 桶，每个桶只放一类东西。

### 背景
S3 的三个基本词：
- **桶**（bucket）：一个仓库。名字**全球唯一**，全世界所有 AWS 账号共用一套名字，所以我们的桶名里带上账号号码：`qafi-186972156153-dev-raw`
- **对象**（object）：桶里的一个文件
- **键**（key）：文件的"路径"，比如 `proteome/v1.0/exhaustive_mutagenesis_data_v1.0.tar.gz`

为什么不用一个大桶放所有东西：**权限和规则是按桶设置的**。原始数据要永久保留、谁都不能改；临时结果 7 天就该删；给 API 用的表要让 API 程序能读。分开放，每个桶一套规则，清清楚楚。

### 用了什么技术，为什么

| 桶 | 放什么 | 规则 | 哪一章用 |
|---|---|---|---|
| `raw` | 原始文件，原样保存、永远不改：全蛋白组压缩包、ClinVar、UniProt 下载 | 永久保留，有版本 | 03 |
| `lake` | 从 Databricks 导出的结果表（全蛋白组预测），给预测 API 查询 | 有版本 | 10 |
| `artifacts` | 导出的模型文件、给 AI 知识库读的文档 | 有版本 | 10、11 |
| `athena` | 草稿纸：10 号用 SQL 查导出的表时，临时结果放这里 | **7 天自动删** | 10 |

**和 Databricks 的关系**：04–09 号的表和模型存在 Databricks 自己的存储里。这 4 个桶放的是 Databricks **之外**的东西。最重要的是 `raw`，它是"原始记录"（system of record）：Databricks 里的表坏了、删了，都能从这里重新生成。

**桶名的规律** `qafi-<账号>-dev-<用途>`：项目、账号、环境（dev 是开发，prod 是正式）、用途。公司里桶名都有这样的规定，看名字就知道它属于谁、干什么。

### 公司里什么情况怎么做
- **要放一类新东西**：先想它的规则（保留多久、谁能读写）和哪个桶一致。不一致就建新桶，不要混放
- **dev 和 prod**：正式环境用另一套桶，通常还在另一个 AWS 账号里，开发时碰不到正式数据。我们只有 dev
- **raw 桶里的文件永远不改**：数据出了新版本，就放到新的键下（比如 `v1.1/`），旧的留着。出了问题，能查到"当时用的是哪一份数据"

### 面试怎么说
> "I separate buckets by purpose — raw as the immutable system of record, lake for curated exports, artifacts for models and documents — because access and retention rules are set per bucket."

---

## 第 2 步 · 给每个桶加保护

### 在干什么
给每个桶开 3 项保护，再建一把我们自己的加密钥匙。

### 背景
云上最常见的数据泄露事故，就是**桶被误设成公开**：任何人知道地址就能下载。另外两类风险：硬盘上的数据没加密；文件被误删、误覆盖后找不回来。医药公司的数据还要满足监管要求：谁、什么时候、改了什么，都要查得到。

### 用了什么技术，为什么

| 保护 | 作用 | 比喻 |
|---|---|---|
| **禁止公开访问**（Block Public Access） | 不管别的设置怎么写，互联网上的人都读不到。00 号在整个账号上开过一次，这里每个桶再开一次，双保险 | 仓库只留员工通道 |
| **加密**（SSE-KMS） | 写进桶的每个文件都用我们的钥匙加密 | 每件货都锁进保险箱 |
| **版本控制**（Versioning） | 文件被覆盖或删除，旧版本还在，可以找回。athena 是草稿纸，不开 | 每次改动都留底稿 |
| **生命周期**（Lifecycle） | 只给 athena：7 天自动删 | 草稿纸定期清理 |

**钥匙放在 KMS**（[基础知识](../../docs/primer_aws_databricks.md) §2.3）：钥匙本身永远不离开 KMS，谁能用这把钥匙可以单独控制，每次使用都有记录。就算有人拿到了文件，没有用钥匙的权限也打不开。4 个桶共用一把钥匙，别名 `alias/qafi-dev-lake`，每月 1 美元。

另外开了 **bucket key**：桶向 KMS 要一把临时的"子钥匙"，用一段时间再换，而不是每个文件都去找一次 KMS。KMS 按使用次数收费，这样费用几乎为零。

### 公司里什么情况怎么做
- **这些保护是默认必须有的**：安全团队会定期自动扫描，没加密、没禁止公开的桶会被标红，要求整改
- **真要把文件分享给外面的人**：不要把桶改成公开。用**临时下载链接**（presigned URL，几分钟后失效），或者通过 API 提供
- **有人误删了文件**：开了版本控制就能从旧版本恢复。所以存重要数据的桶一定要开

### 面试怎么说
> "Every bucket blocks public access, is encrypted with a customer-managed KMS key and keeps versions, so data can't leak, is encrypted at rest, and accidental deletes are recoverable."

---

## 第 3 步 · Terraform：为什么用代码建，而不是在网页上点

### 在干什么
把第 1、2 步的所有东西写在文件里，让 Terraform 去 AWS 上建出来。

### 背景
在 AWS 网页上点几下就能建一个桶。建一两个没问题，但公司里一个平台有成百上千个资源，还有开发、测试、正式好几套环境。靠手点会出现：
- **没有记录**：谁、什么时候、为什么改了这个设置，没人知道
- **没法重建**：环境坏了，或者要再建一套，没人记得当初点了哪些
- **没法审查**：改动生效之前，没人能先看一眼
- **越来越不一致**：开发环境和正式环境慢慢变得不一样

### 用了什么技术，为什么
**Terraform** 是"基础设施即代码"（Infrastructure as Code，IaC）的标准工具。你在文件里写"**应该有什么**"，它负责对比现实，把差的部分补上。

比喻：Terraform 文件是**建筑图纸**，Terraform 是**施工队**。你只改图纸，施工队负责让房子和图纸一致。图纸写的是"应该是什么样"，不是"做什么"，所以运行两次是安全的：第二次发现房子已经和图纸一样，什么都不做。第 5 步你会亲眼看到。

我们的图纸在 `infra/dev/`：

| 文件 | 内容 |
|---|---|
| [`main.tf`](../../infra/dev/main.tf) | 4 个桶、每个桶的保护、1 把钥匙 |
| [`github_oidc.tf`](../../infra/dev/github_oidc.tf) | 给 CI 用的身份（第 4 步） |
| `terraform.tfstate` | Terraform 的**记账本**：图纸上的每一项对应 AWS 里的哪个真实资源。自动生成，**不能提交到 Git**（`.gitignore` 已经排除了） |

打开 `main.tf` 看一眼：每段代码上面都有一行英文注释说明它是干什么的。其中 `resource "aws_s3_bucket"` 加上 `for_each`，一段代码就建出 4 个桶。

**为什么是 Terraform**：它是最常用的 IaC 工具，AWS、Azure、Google Cloud、Databricks、GitHub 都能管。招聘要求里写的 "IaC" 或 "Terraform"，指的就是这一套。

日常只用 3 个命令：

| 命令 | 做什么 | 比喻 |
|---|---|---|
| `terraform init` | 下载 AWS 插件 | 叫施工队 |
| `terraform plan` | **只看不做**：列出将要新建（`+`）、修改（`~`）、删除（`-`）什么 | 施工前核对图纸 |
| `terraform apply` | 真正执行 | 施工 |

### 公司里什么情况怎么做
- **要加或改一个资源**：改 `.tf` 文件 → 提交 Pull Request → CI 自动跑 `plan` → 同事看 plan 的结果 → 合并后 CI 自动 `apply`。平台团队已经搭好了框架，你通常只改几行
- **plan 里出现了你没想到的 `-`（删除）**：**停下，不要 apply**。我们自己就遇到过：改方案时我从代码里删掉了 athena 桶，plan 立刻显示 `4 to destroy`（桶和它的 3 项设置）。因为桶你已经建好了，我们没有 apply，把代码改了回来，plan 又回到 `No changes`
- **有人在网页上手动改了设置**（叫"漂移"，drift）：下次 `plan` 会显示 `~`，要么 apply 改回去，要么把代码改成和现实一致。**Terraform 管的资源，不要在网页上手动改**
- **记账本放哪**：公司里不放在个人电脑上，而是放在一个专门的 S3 桶里，加锁，整个团队共用。我们一个人用，放在本地

### 面试怎么说
> "All infrastructure is defined in Terraform and version-controlled. `plan` previews the change and `apply` executes it — in a team both run in CI, with the plan reviewed in the pull request."

---

## 第 4 步 · 给自动化流程一个身份

### 在干什么
在 AWS 里建一个身份，让 02 号的 GitHub 自动检查（CI）以后能访问这个账号，**而且不存任何密码**。

### 背景
CI 是在 GitHub 的机器上自动运行的检查程序（02 号详细讲）。它以后要往桶里上传数据、部署程序，就需要访问 AWS。程序访问 AWS 用的是**角色**（role），不用账号密码（[基础知识](../../docs/primer_aws_databricks.md) §2.1）。

### 用了什么技术，为什么
`github_oidc.tf` 里建了 3 样东西：

| 资源 | 作用 | 比喻 |
|---|---|---|
| **OIDC 身份提供方** | 告诉 AWS："GitHub 签发的证明，我认" | 认可某家机构盖的章 |
| **IAM 角色** `qafi-dev-github-actions` | CI 进来时扮演的身份，**只允许** `pipi0616/qafi-ml-platform` 这一个仓库扮演 | 只发给指定访客的临时通行证 |
| **权限** | 现在只能看（只读）。03 号再加"往 raw 桶里写"的权限 | 通行证上印的可进区域 |

完整流程在 02 号第 7 步讲。这里只需要记住：**它是 01 号建的，02 号用它**。

### 公司里什么情况怎么做
- **每个仓库、每个环境一个角色**，只给需要的那几项权限（最小权限）
- **CI 报权限不够**：在 Terraform 里给这个角色加上缺的那一项，走 Pull Request 审查，不在网页上手动加

### 面试怎么说
> "CI assumes an IAM role through GitHub's OIDC tokens, restricted to one repository, so no AWS keys are stored anywhere."

---

## 第 5 步 · 执行：init → plan → apply

### 在干什么
让 Terraform 把图纸变成现实。因为你已经建过了，这次它会发现**现实和图纸完全一致**，什么都不用做。这正是我们想看到的。

### 执行
**init**：下载插件。已经下载过了，几秒钟：

In [ ]:
run("terraform -chdir=infra/dev init -input=false -no-color")

**plan**：只看不做。输出里每一行 `Refreshing state...`，是 Terraform 在**去 AWS 上逐个读取真实的设置**，再和图纸比对：

In [ ]:
run("terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan")

### 结果怎么看
最后是 **`No changes. Your infrastructure matches the configuration.`**

意思是：Terraform 刚把 21 个资源的设置（加密、禁止公开、版本控制……）都去 AWS 上读了一遍，和图纸一模一样。这比自己写代码一项项去查更全面。**公司里会定期自动跑 plan，专门用来发现"有人手动改了设置"。**

> 如果是第一次建，这里会显示 `Plan: 21 to add, 0 to change, 0 to destroy.`
> 21 = 钥匙和它的别名（2）+ 4 个桶 + 4 个禁止公开 + 4 个加密 + 3 个版本控制（athena 不开）+ 1 条 7 天自动删 + CI 身份的 3 样东西。

`-out=dev.tfplan` 把这份计划存了下来，下一步执行的**就是你刚看过的这一份**，不会在中间变成别的。

**apply**：执行。这次计划是空的，所以什么都不会改：

In [ ]:
run("terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan")

### 结果怎么看
`Apply complete! Resources: 0 added, 0 changed, 0 destroyed.`

最后，让 Terraform 列出它"公布"的信息。后面每一章都从这里读桶名和角色名，不把它们写死在代码里：

In [ ]:
run("terraform -chdir=infra/dev output -no-color")

### 结果怎么看
- `bucket_names`：4 个桶的名字
- `kms_key_arn`：钥匙的全名。**ARN** 是 AWS 里每个资源独一无二的全名
- `github_actions_role_arn`：CI 的身份，02 号第 7 步会用它

---

## 第 6 步 · 去 AWS 网页上亲眼看一下

### 在干什么
登录 AWS 网页控制台，看看 Terraform 建的东西在 AWS 里长什么样。**只看，不要改。**

### 背景
公司里改东西都通过代码，但**排查问题**时，工程师经常打开网页控制台看现状。认识这几个页面很有用。

### 怎么看
打开 <https://console.aws.amazon.com/>，选 **IAM user**，账号填 `186972156153`，用 `pipi-admin` 登录（就是 `aws login` 时用的那个）。右上角的区域选 **Europe (Frankfurt)**。

| 去哪 | 看什么 |
|---|---|
| 顶部搜索栏输入 **S3** → General purpose buckets | 4 个 `qafi-186972156153-dev-...` 桶 |
| 点 `...-raw` → **Properties** 标签 | Bucket Versioning: **Enabled**；Default encryption: **SSE-KMS**，Bucket Key: **Enabled**；往下的 Tags 里有 `managed_by = terraform` |
| 同一个桶 → **Permissions** 标签 | Block all public access: **On** |
| 点 `...-athena` → **Management** 标签 | 生命周期规则 `expire-query-results` |
| 搜索栏输入 **KMS** → Customer managed keys | 别名 `qafi-dev-lake` 的钥匙 |
| 搜索栏输入 **IAM** → Roles → `qafi-dev-github-actions` → **Trust relationships** 标签 | 只允许 `repo:pipi0616/qafi-ml-platform:*` |

标签 `managed_by = terraform` 是给看到它的人的提示：这个资源由 Terraform 管，不要手动改。

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 四个桶 | S3 | 按用途分开，每个桶一套规则 |
| 2 保护 | 禁止公开、KMS 加密、版本控制、生命周期 | 不泄露、不丢、可追溯 |
| 3 用代码建 | Terraform | 能审查、能重建、能发现漂移 |
| 4 CI 的身份 | IAM 角色 + OIDC | 自动化流程不存密码 |
| 5 先看后做 | `plan` → `apply` | 不会误建、误删 |
| 6 控制台 | AWS 网页 | 排查问题时看现状 |

**以后不想花这 1 美元/月**：`terraform -chdir=infra/dev destroy` 一条命令全删；要用时再 apply，几分钟重建，一模一样。（现在不要删，后面要用。）

**提交代码**：这一章的文件在 02 号第 6 步和其他文件一起第一次提交，这里不用单独提交。

## 面试题

1. **What is infrastructure as code, and why not just use the console?**
   > "Infrastructure is defined in version-controlled code instead of being clicked together. That gives review through pull requests, a history of every change, identical environments and clean teardown. Every AWS resource in my project is in Terraform."

2. **What do `plan` and `apply` do?**
   > "`plan` reads the real infrastructure, compares it with the code and lists what would be created, changed or destroyed, without doing anything. `apply` executes that. In a team, `plan` runs in CI on every pull request and `apply` runs after merge."

3. **What is Terraform state?**
   > "Terraform's record of which real resource belongs to which block of code. Teams keep it in a remote backend such as S3 with locking, so everyone works from the same record. It can contain sensitive values, so it is never committed to git."

4. **How do you secure an S3 bucket?**
   > "Block public access, encrypt with a customer-managed KMS key, enable versioning, and grant access through IAM roles with least privilege. For sharing outside, use short-lived presigned URLs instead of making anything public."

**下一章 02 · 代码库工程化**：让每一次改动都自动检查、自动测试，并把代码第一次放上 GitHub。